# To Be or Not To Be

## Imports and Byte-Pair Encoding
- Using tinyshakespeare dataset
- Strip line breaks
- Lowercase everything

In [12]:
from sre_parse import SPECIAL_CHARS

from torch.xpu import device
!pip install torch tokenizers requests ipykernel

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
text = text.lower()

In [14]:
print(text[:300])

first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us


In [15]:
tokens = list(text)
vocabulary = list(set(tokens))
print("number of tokens in corpus", len(tokens))
print("vocabulary size", len(vocabulary))
print(vocabulary)

number of tokens in corpus 1115394
vocabulary size 39
['&', '-', 'e', 'v', 'n', 'h', ';', 'q', "'", 'z', 'a', 'g', ',', 'y', '$', ':', 'f', 's', '!', 'j', 'c', 'p', 'u', 'x', '3', 'o', 't', 'w', 'k', ' ', '.', 'l', 'i', 'm', 'd', '\n', 'r', '?', 'b']


In [16]:
vocab_size = 1_000
def byte_pair_encode(tokens, vocabulary, vocab_size=1_000):
    """
    Isaac: I copied and pasted this implementation of BPE from ChatGPT.
    It uses recursion with a base case of returning the vocabulary once it's matched the vocab size.
    For the "recursive step" (idk proper, kinda like inductivive step in math),
    it adds the most common pair of tokens to the vocab and merges them in the corpus.
    I had a buggy implementation of the same idea that I had GPT 5.6 Sol (Medium) correct.

    It was annoyingly slow though, so I just ended up using BPE from HuggingFace, which is in Rust. :D
    """
    # Base case
    if len(vocabulary) >= vocab_size:
        return tokens, vocabulary

    # Count adjacent pairs
    count_by_token_pair = {}

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        count_by_token_pair[pair] = count_by_token_pair.get(pair, 0) + 1

    # Find most common pair
    most_used_pair = max(
        count_by_token_pair,
        key=count_by_token_pair.get
    )

    # Add new token
    new_token = most_used_pair[0] + most_used_pair[1]
    vocabulary.append(new_token)

    # Merge occurrences
    new_tokens = []
    i = 0

    while i < len(tokens):
        if (
            i < len(tokens) - 1
            and (tokens[i], tokens[i + 1]) == most_used_pair
        ):
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    # Return recursion
    return byte_pair_encode(new_tokens, vocabulary, vocab_size)

# tokens, vocabulary = byte_pair_encode(tokens, vocabulary, vocab_size)

In [17]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

In [19]:
import re
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

"""
I'm using HF's BPE (implemented in Rust) because I don't want to wait
for so long every time I run this program from scratch.

Felix: splits into sentences before training so BPE runs on many short
strings instead of one giant one (much faster). Speakers are dropped.
"""

def byte_pair_encode_fast(tokens, vocabulary, vocab_size=1_000):
    text = "".join(tokens)

    # Split into sentences: drop blank lines and speaker labels, keep punctuation
    lines = [l.strip() for l in text.split("\n")
             if l.strip() and not l.rstrip().endswith(":")]
    joined = " ".join(lines)
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", joined) if s.strip()]

    tokenizer = Tokenizer(BPE())

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        special_tokens=["<START>", "<STOP>"],
        initial_alphabet=vocabulary,
    )

    tokenizer.train_from_iterator(sentences, trainer=trainer)

    # BPE-tokenized corpus: one list of IDs per sentence, wrapped in START/STOP
    START = tokenizer.token_to_id("<START>")
    STOP = tokenizer.token_to_id("<STOP>")
    sentence_ids = [[START] + enc.ids + [STOP] for enc in tokenizer.encode_batch(sentences)]

    # Vocabulary as a list ordered by token ID
    vocab_dict = tokenizer.get_vocab()
    vocabulary = [token for token, idx in sorted(vocab_dict.items(), key=lambda x: x[1])]

    return sentence_ids, vocabulary, tokenizer, sentences


sentence_ids, vocabulary, tokenizer, sentences = byte_pair_encode_fast(
    tokens,
    vocabulary,
    vocab_size
)
tokenizer.save("bpe_tokenizer.json")

print(len(sentences), "sentences |", len(vocabulary), "vocab")
print(sentences[0])
print(tokenizer.encode(sentences[0]).tokens)




12221 sentences | 1000 vocab
before we proceed any further, hear me speak.
['before ', 'we ', 'pro', 'ce', 'ed ', 'any ', 'f', 'ur', 'ther', ', ', 'hear', ' me ', 'speak', '.']


In [20]:
"""
Isaac:
Saving the python objects to file before loading them back up
.pkl is the extention for python objects.
wb = write binary, rb = read binary
"""

import pickle

with open("bpe_data.pkl", "wb") as f:
    pickle.dump((tokens, vocabulary), f)

In [21]:
with open("bpe_data.pkl", "rb") as f:
    tokens, vocabulary = pickle.load(f)

## Create Dataset
Consider this the function comment for the cell below from Isaac
- Training pairs (token[i], token[i+k]) -> k
- For i going from 0 to len(token)-k
- For k going from 0 to 10
- We predict word distance since it might actually matter
- I'm using Pytorch since I like its style and have some experience with it

In [22]:
import torch
from torch.utils.data import TensorDataset

def create_dataset(tokens, token_to_idx, max_distance=10):
    X = []
    y = []

    for k in range(1, max_distance + 1):
        for i in range(len(tokens) - k):
            X.append([
                token_to_idx[tokens[i]],
                token_to_idx[tokens[i + k]]
            ])
            y.append(k)

    X = torch.tensor(X, dtype=torch.long)
    y = torch.tensor(y, dtype=torch.long)

    return TensorDataset(X, y)

In [23]:
# Isaac: Making a dictionary to get the indice of each token (for future one-hot encoding)
token_to_idx = {token: i for i, token in enumerate(vocabulary)}

"""
Creating a train-validation-test split
"""

from torch.utils.data import random_split, DataLoader

dataset = create_dataset(tokens, token_to_idx)

# GPT wrote this comment, but I set the values: make a split of 70% train, 10% validation, 20% test
train_size = int(0.7 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

torch.manual_seed(0)
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
)

# Using data loader to convert to nice format for loading, training, and evaluating (with batches for arrellization)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Train the Embedder

Consider this the function comment for the cell below from Isaac
- Embed token[i] and embed token[i+k]: shape vocab_size by 128 (embedding dimension)
- Compute the dot product of the embeded tokens (ideally it's k).
- Do gradient descent from the ideal value of k (based on MSE loss)
- Notice how we actually just need to train the embedding

In [29]:
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [30]:
embedding_dim = 128

# Shape conceptually: (vocab_size, embedding_dim)
embedding = nn.Embedding(vocab_size, embedding_dim).to(device)

optimizer = torch.optim.Adam(embedding.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(12):
    # Training
    embedding.train()

    for batch_idx, (X_batch, k_batch) in enumerate(train_loader):
        if batch_idx >= 16_384:
            break  # Isaac: It was taking too long so I decided to cut the training data short

        X_batch = X_batch.to(device)
        k_batch = k_batch.to(device)
        # Embed token[i] and token[i+k]
        e1 = embedding(X_batch[:, 0])  # (batch_size, embedding_dim)
        e2 = embedding(X_batch[:, 1])  # (batch_size, embedding_dim)

        # Dot product for each pair
        predicted_k = (e1 * e2).sum(dim=1)  # (batch_size,)

        # Ideally predicted_k == k
        loss = loss_fn(predicted_k, k_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation
    embedding.eval()

    total_val_loss = 0
    total_val_examples = 0

    with torch.no_grad():
        for X_batch, k_batch in val_loader:

            X_batch = X_batch.to(device)
            k_batch = k_batch.to(device)

            e1 = embedding(X_batch[:, 0])
            e2 = embedding(X_batch[:, 1])

            predicted_k = (e1 * e2).sum(dim=1)

            val_loss = loss_fn(predicted_k, k_batch.float())

            batch_size = X_batch.size(0)
            total_val_loss += val_loss.item() * batch_size
            total_val_examples += batch_size

    average_val_loss = total_val_loss / total_val_examples

    print(
        f"Epoch {epoch}: "
        f"train loss = {loss.item():.4f}, "
        f"validation loss = {average_val_loss:.4f}"
    )

    """
    Isaac, based on ChatGPT's notes:
    train loss is based on average MSE loss for last batch
    while val loss is for average over all val examples
    """

Epoch 0: train loss = 7.0901, validation loss = 8.4949
Epoch 1: train loss = 7.7239, validation loss = 8.2481
Epoch 2: train loss = 8.1489, validation loss = 8.2004
Epoch 3: train loss = 9.5082, validation loss = 8.1776
Epoch 4: train loss = 9.1374, validation loss = 8.1629
Epoch 5: train loss = 7.2627, validation loss = 8.1730
Epoch 6: train loss = 7.5505, validation loss = 8.1593
Epoch 7: train loss = 9.1026, validation loss = 8.1539
Epoch 8: train loss = 8.2436, validation loss = 8.1555
Epoch 9: train loss = 11.1569, validation loss = 8.1555
Epoch 10: train loss = 8.0340, validation loss = 8.1538
Epoch 11: train loss = 7.5767, validation loss = 8.1433


In [31]:
"""
Isaac: Evaluate the embedding matrix on the test set to see if the MSE error is similar
"""

embedding.eval()

total_loss = 0
total_examples = 0

with torch.no_grad():
    for X_batch, k_batch in test_loader:
        X_batch = X_batch.to(device)
        k_batch = k_batch.to(device)
        e1 = embedding(X_batch[:, 0])
        e2 = embedding(X_batch[:, 1])

        predicted_k = (e1 * e2).sum(dim=1)

        loss = loss_fn(predicted_k, k_batch.float())

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

test_loss = total_loss / total_examples

print(f"Test MSE: {test_loss:.4f}")

Test MSE: 8.1462


## Making a Next-token Predictor

In [32]:
SENT_DELIMITERS = [".","!","?"]

# Felix: I'm leaving out speakers to keep the data simpler
import re

# Felix: I'm leaving out speakers to keep the data simpler
lines = [l.strip() for l in text.split("\n")
         if l.strip() and not l.rstrip().endswith(":")]   # drop blank lines + speaker labels

joined = " ".join(lines)
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", joined) if s.strip()]

print(len(sentences))
print(sentences[:5])

12221
['before we proceed any further, hear me speak.', 'speak, speak.', 'you are all resolved rather to die than to famish?', 'resolved.', 'resolved.']


In [37]:
"""
Felix: Claude dataset gen cell. added start tokens
"""
WINDOW = 8
START = tokenizer.token_to_id("<START>")

def make_next_token_data(sents, window=WINDOW):
    X, y = [], []
    for ids in sents:
        padded = [START] * (window - 1) + ids        # ids already begins with <START>
        for t in range(window, len(padded)):
            X.append(padded[t - window:t])
            y.append(padded[t])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Split by sentence, in order: no sentence appears in two splits
n = len(sentence_ids)
tr_end, val_end = int(0.7 * n), int(0.8 * n)

X_tr, y_tr = make_next_token_data(sentence_ids[:tr_end])
X_val, y_val = make_next_token_data(sentence_ids[tr_end:val_end])
X_te, y_te = make_next_token_data(sentence_ids[val_end:])

next_train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=256, shuffle=True)
next_val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=1024)
next_test_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=1024)

print(X_tr.shape, y_tr.shape)

torch.Size([247369, 8]) torch.Size([247369])


In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F

"""
Felix: had Claude translate this network from Keras into Pytorch

next_word = keras.Sequential([
    keras.Input(shape=(4,), dtype="int32"),
    layers.Embedding(VOCAB_SIZE, 100,
                     embeddings_initializer=keras.initializers.Constant(emb),
                     trainable=False),
    layers.Flatten(),
    layers.Dense(256, activation="relu",
                 kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dropout(0.4, name='dropout_1'),
      layers.Dense(128, activation="relu",
                kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dropout(0.4, name='dropout_2'),
    layers.Dense(64, activation="relu",
                 kernel_regularizer=keras.regularizers.l1(1e-3)),
    layers.Dropout(0.4, name='dropout_3'),
    layers.Dense(VOCAB_SIZE, activation="softmax")
])

next_word.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model and adjustments beyond the translation are mine.
"""

class NextTokenPredictor(nn.Module):
    def __init__(self, emb, window=8):
        super().__init__()
        vocab_size, emb_dim = emb.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.as_tensor(emb, dtype=torch.float32), freeze=False)
        self.pos_logits = nn.Parameter(torch.zeros(window))
        self.fc1 = nn.LazyLinear(256) # swapped to lazy linear to infer length
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.out = nn.Linear(128, vocab_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):                        # x: (B, window) token IDs
        e = self.embedding(x)                    # (B, window, emb_dim)
        w = torch.softmax(self.pos_logits, 0)
        h = (e * w[None, :, None]).sum(dim=1)    # weighted average -> (B, emb_dim)
        h = self.dropout(F.relu(self.fc1(h)))    # emb_dim -> 256
        h = self.dropout(F.relu(self.fc2(h)))    # 256 -> 128
        # h = self.dropout(F.relu(self.fc3(h)))    # 128 -> 64
        return self.out(h)                       # 64 -> vocab_size

    def reg_penalty(self):
        # Keras kernel_regularizer: penalty on weights only (not biases), added to loss
        return (1e-3 * self.fc1.weight.pow(2).sum()
                + 1e-3 * self.fc2.weight.pow(2).sum()
                + 1e-3 * self.fc3.weight.abs().sum())   # the l1 layer


In [52]:
emb = embedding.weight.detach()
model = NextTokenPredictor(emb).to(device)
model(next(iter(next_train_loader))[0].to(device))    # initializes fc1
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-3)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=1e-3)

def run_epoch(loader, train):
    model.train(train)
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = F.cross_entropy(logits, y)          # sparse_categorical_crossentropy
            if train:
                optimizer.zero_grad()
                (loss).backward() # removed + model.reg_penalty() term to try and improve performance.
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
for epoch in range(100):
    tl, ta = run_epoch(next_train_loader, train=True)
    vl, va = run_epoch(next_val_loader, train=False)
    for k, v in zip(history, (tl, ta, vl, va)):
        history[k].append(v)
    print(f"epoch {epoch}: loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f}")

epoch 0: loss 5.9299 acc 0.0501 | val_loss 5.3067 val_acc 0.0796
epoch 1: loss 4.9606 acc 0.0988 | val_loss 4.7987 val_acc 0.1089
epoch 2: loss 4.6456 acc 0.1155 | val_loss 4.6385 val_acc 0.1197
epoch 3: loss 4.5058 acc 0.1230 | val_loss 4.5563 val_acc 0.1263
epoch 4: loss 4.4144 acc 0.1308 | val_loss 4.5069 val_acc 0.1318
epoch 5: loss 4.3449 acc 0.1371 | val_loss 4.4726 val_acc 0.1370
epoch 6: loss 4.2911 acc 0.1432 | val_loss 4.4363 val_acc 0.1407
epoch 7: loss 4.2432 acc 0.1486 | val_loss 4.4141 val_acc 0.1448
epoch 8: loss 4.2058 acc 0.1535 | val_loss 4.4007 val_acc 0.1481
epoch 9: loss 4.1687 acc 0.1580 | val_loss 4.3819 val_acc 0.1522
epoch 10: loss 4.1394 acc 0.1617 | val_loss 4.3676 val_acc 0.1550
epoch 11: loss 4.1107 acc 0.1646 | val_loss 4.3698 val_acc 0.1551
epoch 12: loss 4.0883 acc 0.1680 | val_loss 4.3532 val_acc 0.1551
epoch 13: loss 4.0636 acc 0.1705 | val_loss 4.3521 val_acc 0.1567
epoch 14: loss 4.0448 acc 0.1722 | val_loss 4.3423 val_acc 0.1596
epoch 15: loss 4.026

In [53]:
torch.save(model.state_dict(), "next_token_model.pt")

In [58]:
"""
Felix: generation loop with temperature, translated by Claude and taken from my other Shakespeare project
"""

START = tokenizer.token_to_id("<START>")
STOP = tokenizer.token_to_id("<STOP>")
@torch.no_grad()
def generate(seed="", max_tokens=60, min_tokens=5, temperature=0.8, k=20):
    model.eval()
    ids = [START] + tokenizer.encode(seed.lower()).ids   # corpus was lowercased
    generated = []

    for i in range(max_tokens):
        ctx = ([START] * WINDOW + ids)[-WINDOW:]          # pad with START, matching training
        x = torch.tensor([ctx], dtype=torch.long, device=device)
        logits = model(x)[0].float()

        logits[START] = float("-inf")                     # never generate START
        if i < min_tokens:
            logits[STOP] = float("-inf")                  # don't stop too early

        top_vals, top_idx = torch.topk(logits, k)         # keep the k best
        probs = torch.softmax(top_vals / temperature, dim=0)
        next_id = top_idx[torch.multinomial(probs, 1)].item()

        if next_id == STOP:
            break
        ids.append(next_id)
        generated.append(next_id)

    seed_text = "".join(tokenizer.id_to_token(t) for t in ids[1:len(ids) - len(generated)])
    gen_text = "".join(tokenizer.id_to_token(t) for t in generated)
    return seed_text + gen_text

for _ in range(3):
    print(generate("my lord", temperature=0.7))

my lord; for, if my soul, in thy palm to a most remain, whilst wrong; you cannot?
my lord; and that i may be not the crown, which are injurious to our prince!
my lord: it is we in thy face into our heads.


## Citation of AI Use
I used this ChatGPT chat for syntax while I prompted with my ideas: https://chatgpt.com/share/6abaeddf-4c24-83ea-9777-f0f85be4d542

All of my comments have my name on them. Otherwise, ChatGPT wrote the small comment for clarity.